In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# ==========================================
# تنظیمات اولیه
# ==========================================

np.random.seed(42)
random.seed(42)

# 1. وزن واقعی شعبه‌ها (تعداد رکورد متفاوت)
branches_weight = {
    "تهران_مرکز": 2800,
    "تهران_غرب": 2200,
    "مشهد": 1800,
    "اصفهان": 1500,
    "شیراز": 1200,
    "تبریز": 1000,
    "اهواز": 800,
    "رشت": 700
}

# 2. Category برای محصولات
product_categories = {
    "یخچال": {"category": "سرمایشی", "margin": 0.18, "price": 45000000},
    "فریزر": {"category": "سرمایشی", "margin": 0.17, "price": 35000000},
    "کولر گازی": {"category": "سرمایشی", "margin": 0.14, "price": 42000000},
    "ماشین لباسشویی": {"category": "شستشو", "margin": 0.15, "price": 30000000},
    "ماشین ظرفشویی": {"category": "شستشو", "margin": 0.16, "price": 38000000},
    "جاروبرقی": {"category": "نظافت", "margin": 0.20, "price": 9000000},
    "تلویزیون": {"category": "صوتی تصویری", "margin": 0.12, "price": 25000000},
    "مایکروویو": {"category": "پخت و پز", "margin": 0.18, "price": 12000000},
    "اجاق گاز": {"category": "پخت و پز", "margin": 0.16, "price": 18000000},
    "آبگرمکن": {"category": "گرمایشی", "margin": 0.17, "price": 14000000},
    "پکیج": {"category": "گرمایشي", "margin": 0.15, "price": 25000000},
    "بخاری برقی": {"category": "گرمایشی", "margin": 0.18, "price": 8000000}
}

# 3. الگوی فروش فصلی و منطقه‌ای
seasonal_boost = {
    "سرمایشی": {"summer": 2.5, "winter": 0.6, "spring_fall": 1.0},
    "گرمایشی": {"summer": 0.4, "winter": 2.8, "spring_fall": 1.0},
    "شستشو": {"summer": 1.2, "winter": 1.1, "spring_fall": 1.0},
    "نظافت": {"summer": 1.0, "winter": 1.0, "spring_fall": 1.0},
    "صوتی تصویری": {"summer": 1.1, "winter": 1.3, "spring_fall": 1.0},
    "پخت و پز": {"summer": 0.9, "winter": 1.2, "spring_fall": 1.0}
}

# توزیع محصول بر اساس منطقه
regional_product_preference = {
    "تهران_مرکز": {"سرمایشی": 0.3, "شستشو": 0.25, "صوتی تصویری": 0.2, "پخت و پز": 0.15, "نظافت": 0.05, "گرمایشی": 0.05},
    "تهران_غرب": {"سرمایشی": 0.28, "شستشو": 0.27, "صوتی تصویری": 0.2, "پخت و پز": 0.15, "نظافت": 0.05, "گرمایشی": 0.05},
    "مشهد": {"سرمایشی": 0.25, "شستشو": 0.25, "صوتی تصویری": 0.15, "پخت و پز": 0.15, "نظافت": 0.1, "گرمایشی": 0.1},
    "اصفهان": {"سرمایشی": 0.3, "شستشو": 0.25, "صوتی تصویری": 0.15, "پخت و پز": 0.15, "نظافت": 0.05, "گرمایشی": 0.1},
    "شیراز": {"سرمایشی": 0.35, "شستشو": 0.25, "صوتی تصویری": 0.15, "پخت و پز": 0.1, "نظافت": 0.05, "گرمایشی": 0.1},
    "تبریز": {"گرمایشی": 0.45, "شستشو": 0.2, "سرمایشی": 0.1, "صوتی تصویری": 0.1, "پخت و پز": 0.1, "نظافت": 0.05},
    "اهواز": {"سرمایشی": 0.6, "شستشو": 0.15, "صوتی تصویری": 0.1, "پخت و پز": 0.1, "نظافت": 0.03, "گرمایشی": 0.02},
    "رشت": {"گرمایشی": 0.35, "شستشو": 0.25, "سرمایشی": 0.1, "صوتی تصویری": 0.1, "پخت و پز": 0.15, "نظافت": 0.05}
}

default_pref = {"سرمایشی": 0.3, "گرمایشی": 0.15, "شستشو": 0.2, "صوتی تصویری": 0.15, "پخت و پز": 0.15, "نظافت": 0.05}

# Customer Segment
customer_segments = {
    "Retail": 0.55,
    "Wholesale": 0.15,
    "Corporate": 0.05,
    "Online": 0.25
}

# ==========================================
# توابع کمکی اصلاح شده
# ==========================================

def get_season(date):
    """تشخیص فصل بر اساس ماه (ورودی pandas Timestamp)"""
    month = date.month
    if month in [6, 7, 8]:
        return "summer"
    elif month in [12, 1, 2]:
        return "winter"
    else:
        return "spring_fall"

def add_dirty_data(df, branch_name):
    """اضافه کردن داده‌های خراب عمدی"""
    df_copy = df.copy()
    n_rows = len(df_copy)
    
    # 1. مقادیر گمشده (1%)
    missing_idx = np.random.choice(df_copy.index, size=int(n_rows * 0.01), replace=False)
    df_copy.loc[missing_idx, "Quantity"] = np.nan
    
    # 2. سطرهای تکراری (0.5%)
    dup_idx = np.random.choice(df_copy.index, size=int(n_rows * 0.005), replace=False)
    if len(dup_idx) > 0:
        duplicates = df_copy.loc[dup_idx].copy()
        df_copy = pd.concat([df_copy, duplicates], ignore_index=True)
    
    # 3. تاریخ اشتباه (0.5%)
    bad_date_idx = np.random.choice(df_copy.index, size=int(n_rows * 0.005), replace=False)
    for idx in bad_date_idx:
        if random.random() > 0.5:
            df_copy.loc[idx, "Date"] = datetime.now() + timedelta(days=random.randint(30, 365))
        else:
            df_copy.loc[idx, "Date"] = datetime(2010, random.randint(1,12), random.randint(1,28))
    
    # 4. قیمت منفی (0.2%)
    neg_idx = np.random.choice(df_copy.index, size=int(n_rows * 0.002), replace=False)
    df_copy.loc[neg_idx, "Unit_Price"] = -abs(df_copy.loc[neg_idx, "Unit_Price"])
    
    return df_copy

# ==========================================
# تولید دیتا برای یک شعبه
# ==========================================

def generate_branch_data(branch_name, n_rows):
    """تولید داده واقعی برای یک شعبه"""
    
    if branch_name in regional_product_preference:
        cat_dist = regional_product_preference[branch_name]
    else:
        cat_dist = default_pref
    
    category_products = {}
    for prod, info in product_categories.items():
        cat = info["category"]
        if cat not in category_products:
            category_products[cat] = []
        category_products[cat].append(prod)
    
    data = []
    start_date = pd.Timestamp("2022-01-01")
    end_date = pd.Timestamp("2025-12-31")
    date_range = pd.date_range(start_date, end_date, freq="D")
    
    for order_id in range(1, n_rows + 1):
        
        category = np.random.choice(list(cat_dist.keys()), p=list(cat_dist.values()))
        
        # انتخاب محصول از دسته
        if category in category_products and len(category_products[category]) > 0:
            product = np.random.choice(category_products[category])
        else:
            product = list(product_categories.keys())[0]
        
        product_info = product_categories[product]
        
        # انتخاب تاریخ و تبدیل به Timestamp
        random_date = np.random.choice(date_range)
        order_date = pd.Timestamp(random_date)
        season = get_season(order_date)
        
        seasonal_factor = seasonal_boost.get(category, {}).get(season, 1.0)
        
        base_price = product_info["price"]
        price_variation = np.random.uniform(0.85, 1.15)
        unit_price = int(base_price * price_variation)
        
        base_quantity = np.random.randint(1, 5)
        quantity = max(1, int(base_quantity * seasonal_factor * np.random.uniform(0.7, 1.3)))
        
        customer_type = np.random.choice(list(customer_segments.keys()), p=list(customer_segments.values()))
        
        # تخفیف بر اساس نوع مشتری
        if customer_type == "Corporate":
            discount_pct = np.random.choice([10, 15, 20, 25], p=[0.3, 0.3, 0.25, 0.15])
        elif customer_type == "Wholesale":
            discount_pct = np.random.choice([5, 10, 15, 20], p=[0.2, 0.35, 0.3, 0.15])
        elif customer_type == "Online":
            discount_pct = np.random.choice([0, 5, 10], p=[0.5, 0.3, 0.2])
        else:
            discount_pct = np.random.choice([0, 5, 10], p=[0.6, 0.3, 0.1])
        
        gross_sales = unit_price * quantity
        discount_amount = gross_sales * discount_pct / 100
        revenue = gross_sales - discount_amount
        margin = product_info["margin"]
        cost = revenue * (1 - margin)
        profit = revenue - cost
        
        data.append({
            "Order_ID": f"{branch_name[:4]}-{order_id:05d}",
            "Date": order_date,
            "Branch": branch_name,
            "Category": category,
            "Product": product,
            "Quantity": quantity,
            "Unit_Price": unit_price,
            "Discount_Percent": discount_pct,
            "Revenue": revenue,
            "Cost": round(cost),
            "Profit": round(profit),
            "Customer_Segment": customer_type
        })
    
    df = pd.DataFrame(data)
    df = df.sort_values("Date").reset_index(drop=True)
    df = add_dirty_data(df, branch_name)
    
    return df

# ==========================================
# ایجاد پوشه خروجی
# ==========================================

output_folder = "branch_data_advanced"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"✅ پوشه '{output_folder}' ساخته شد")

# ==========================================
# تولید فایل برای هر شعبه
# ==========================================

print("\n" + "="*70)
print("🚀 شروع تولید دیتاست پیشرفته فروش لوازم خانگی")
print("="*70)

all_branches_summary = []

for branch_name, n_rows in branches_weight.items():
    print(f"\n📊 در حال تولید داده برای شعبه: {branch_name} ({n_rows} رکورد)")
    
    df_branch = generate_branch_data(branch_name, n_rows)
    
    filename = f"{output_folder}/{branch_name}_sales.xlsx"
    df_branch.to_excel(filename, index=False)
    
    total_rev = df_branch["Revenue"].sum()
    total_profit = df_branch["Profit"].sum()
    missing_quantity = df_branch["Quantity"].isna().sum()
    duplicate_count = df_branch.duplicated().sum()
    
    summary = {
        "Branch": branch_name,
        "Records": len(df_branch),
        "Total Revenue": total_rev,
        "Total Profit": total_profit,
        "Missing Values": missing_quantity,
        "Duplicate Rows": duplicate_count
    }
    all_branches_summary.append(summary)
    
    print(f"   ✅ ذخیره شد: {filename}")
    print(f"   📈 فروش کل: {total_rev:,.0f} تومان")
    print(f"   💰 سود کل: {total_profit:,.0f} تومان")
    print(f"   ⚠️ Missing: {missing_quantity} | Duplicate: {duplicate_count}")

# ==========================================
# فایل خلاصه
# ==========================================

print("\n" + "="*70)
print("📊 ایجاد فایل خلاصه کلی")
print("="*70)

summary_df = pd.DataFrame(all_branches_summary)
summary_file = f"{output_folder}/00_Master_Summary.xlsx"
summary_df.to_excel(summary_file, index=False)

print("\n📋 خلاصه نهایی:")
print(summary_df.to_string(index=False))

print("\n" + "="*70)
print(f"✅ همه فایل‌ها در پوشه '{output_folder}' ذخیره شدند.")
print("\n📁 لیست فایل‌های تولید شده:")
for file in os.listdir(output_folder):
    file_path = os.path.join(output_folder, file)
    size = os.path.getsize(file_path) / 1024
    print(f"   📄 {file} ({size:.1f} KB)")
print("="*70)

✅ پوشه 'branch_data_advanced' ساخته شد

🚀 شروع تولید دیتاست پیشرفته فروش لوازم خانگی

📊 در حال تولید داده برای شعبه: تهران_مرکز (2800 رکورد)


C:\Users\Vivobook\AppData\Local\Temp\ipykernel_23048\2680850083.py:180: RuntimeWarning: overflow encountered in scalar multiply
  discount_amount = gross_sales * discount_pct / 100


   ✅ ذخیره شد: branch_data_advanced/تهران_مرکز_sales.xlsx
   📈 فروش کل: 206,338,405,338 تومان
   💰 سود کل: 32,242,073,357 تومان
   ⚠️ Missing: 28 | Duplicate: 14

📊 در حال تولید داده برای شعبه: تهران_غرب (2200 رکورد)


C:\Users\Vivobook\AppData\Local\Temp\ipykernel_23048\2680850083.py:180: RuntimeWarning: overflow encountered in scalar multiply
  discount_amount = gross_sales * discount_pct / 100


   ✅ ذخیره شد: branch_data_advanced/تهران_غرب_sales.xlsx
   📈 فروش کل: 158,560,338,156 تومان
   💰 سود کل: 24,616,461,077 تومان
   ⚠️ Missing: 23 | Duplicate: 11

📊 در حال تولید داده برای شعبه: مشهد (1800 رکورد)


...

   ✅ ذخیره شد: branch_data_advanced/تبریز_sales.xlsx
   📈 فروش کل: 51,884,507,881 تومان
   💰 سود کل: 8,252,127,469 تومان
   ⚠️ Missing: 10 | Duplicate: 5

📊 در حال تولید داده برای شعبه: اهواز (800 رکورد)


C:\Users\Vivobook\AppData\Local\Temp\ipykernel_23048\2680850083.py:180: RuntimeWarning: overflow encountered in scalar multiply
  discount_amount = gross_sales * discount_pct / 100


   ✅ ذخیره شد: branch_data_advanced/اهواز_sales.xlsx
   📈 فروش کل: 67,481,290,606 تومان
   💰 سود کل: 10,715,352,270 تومان
   ⚠️ Missing: 8 | Duplicate: 4

📊 در حال تولید داده برای شعبه: رشت (700 رکورد)


C:\Users\Vivobook\AppData\Local\Temp\ipykernel_23048\2680850083.py:180: RuntimeWarning: overflow encountered in scalar multiply
  discount_amount = gross_sales * discount_pct / 100


   ✅ ذخیره شد: branch_data_advanced/رشت_sales.xlsx
   📈 فروش کل: 36,104,925,402 تومان
   💰 سود کل: 5,722,645,259 تومان
   ⚠️ Missing: 7 | Duplicate: 3

📊 ایجاد فایل خلاصه کلی

📋 خلاصه نهایی:
    Branch  Records  Total Revenue  Total Profit  Missing Values  Duplicate Rows
تهران_مرکز     2814   2.063384e+11   32242073357              28              14
 تهران_غرب     2211   1.585603e+11   24616461077              23              11
      مشهد     1809   1.199407e+11   18967310658              18               9
    اصفهان     1507   9.801766e+10   15471825530              15               7
     شیراز     1206   9.219147e+10   14508068303              12               6
     تبریز     1005   5.188451e+10    8252127469              10               5
     اهواز      804   6.748129e+10   10715352270               8               4
       رشت      703   3.610493e+10    5722645259               7               3

✅ همه فایل‌ها در پوشه 'branch_data_advanced' ذخیره شدند.

📁 لیست فایل‌های تولید